In [1]:
import numpy as np
import pandas as pd
import torch 
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

data=load_breast_cancer()
x,y=data.data,data.target

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

x_train_tensor=torch.tensor(x_train_scaled,dtype=torch.float32)
x_test_tensor=torch.tensor(x_test_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.float32).view(-1,1)
y_test_tensor=torch.tensor(y_test,dtype=torch.float32).view(-1,1)

train_dataset=TensorDataset(x_train_tensor,y_train_tensor)
train_loader=DataLoader(train_dataset,batch_size=1,shuffle=True)

class ANN(nn.Module):

    def __init__(self,input_features):
        super().__init__()

        self.network=nn.Sequential(nn.Linear(input_features,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,1))

    def forward(self,x):
        return self.network(x)

model=ANN(x_train.shape[1])
print(model)                

ANN(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [2]:
criterion=nn.BCEWithLogitsLoss()
optimizer=torch.optim.SGD(model.parameters(),lr=0.01)

epochs=10

for epoch in range(epochs):
    model.train()

    total_loss=0
    for x_batch,y_batch in train_loader:
        outputs=model(x_batch)
        loss=criterion(outputs,y_batch)

        optimizer.zero_grad()
        loss.backward()

        optimizer.step()

        total_loss+=loss.item()
    avg_loss=total_loss/len(train_loader)
    print( f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {avg_loss:.4f}")

Epoch [1/10], Loss: 0.4076
Epoch [2/10], Loss: 0.1165
Epoch [3/10], Loss: 0.0739
Epoch [4/10], Loss: 0.0588
Epoch [5/10], Loss: 0.0511
Epoch [6/10], Loss: 0.0449
Epoch [7/10], Loss: 0.0411
Epoch [8/10], Loss: 0.0373
Epoch [9/10], Loss: 0.0341
Epoch [10/10], Loss: 0.0312


In [3]:
model.eval()
with torch.no_grad():
    logits=model(x_test_tensor)
    probab=torch.sigmoid(logits)
    predictions=(probab>=0.5).float()

accuracy=(predictions.eq(y_test_tensor).sum().item()/len(y_test_tensor))
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9474
